In [0]:
from pyspark.sql.functions import (
    col,
    when,
    count,
    current_timestamp
)

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType
)

In [0]:
catalog_name = "customer_returns"

In [0]:
schema = StructType([
    StructField("response_id", StringType(), True),
    StructField("age_group", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("purchase_frequency", StringType(), True),
    StructField("ever_returned", StringType(), True),
    StructField("return_count", IntegerType(), True),
    StructField("main_return_reason", StringType(), True),
    StructField("wanted_exchange", StringType(), True),
    StructField("exchange_available", StringType(), True),
    StructField("replacement_unavailable", StringType(), True),
    StructField("what_happened_when_unavailable", StringType(), True)
])

In [0]:
raw_path = "/Volumes/customer_returns/source_data/raw/rawdata.csv"

In [0]:
df_bronze = (
    spark.read
    .format("csv")
    .option("header", True)
    .schema(schema)
    .load(raw_path)
    .withColumn("ingested_at", current_timestamp())
)


In [0]:
df_bronze.show(10, truncate=False)

+-----------+---------+----------+-------------------+-------------+------------+------------------+---------------+------------------+-------------------------+------------------------------+--------------------------+
|response_id|age_group|gender    |purchase_frequency |ever_returned|return_count|main_return_reason|wanted_exchange|exchange_available|replacement_unavailable  |what_happened_when_unavailable|ingested_at               |
+-----------+---------+----------+-------------------+-------------+------------+------------------+---------------+------------------+-------------------------+------------------------------+--------------------------+
|1          |46-55    |Female    |Once every 3 months|Yes          |1           |Wrong Size        |Yes            |No                |No Stock in Desired Size |Returned the Product          |2026-08-26 14:19:55.471063|
|2          |26-35    |Non-binary|2-3 times a month  |Yes          |2           |Wrong Size        |Yes            |No  

In [0]:
print("Total rows:", df_bronze.count())
print("Total columns:", len(df_bronze.columns))

Total rows: 150
Total columns: 12


In [0]:
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(
        "customer_returns.bronze.brz_customer_returns"
    )

In [0]:
df_bronze_delta = spark.table(
    "customer_returns.bronze.brz_customer_returns"
)

df_bronze_delta.show(10, truncate=False)

+-----------+---------+----------+-------------------+-------------+------------+------------------+---------------+------------------+-------------------------+------------------------------+--------------------------+
|response_id|age_group|gender    |purchase_frequency |ever_returned|return_count|main_return_reason|wanted_exchange|exchange_available|replacement_unavailable  |what_happened_when_unavailable|ingested_at               |
+-----------+---------+----------+-------------------+-------------+------------+------------------+---------------+------------------+-------------------------+------------------------------+--------------------------+
|1          |46-55    |Female    |Once every 3 months|Yes          |1           |Wrong Size        |Yes            |No                |No Stock in Desired Size |Returned the Product          |2026-08-26 14:19:57.676706|
|2          |26-35    |Non-binary|2-3 times a month  |Yes          |2           |Wrong Size        |Yes            |No  

In [0]:
print("Bronze rows:", df_bronze_delta.count())

Bronze rows: 150


In [0]:
df_bronze = spark.table(
    "customer_returns.bronze.brz_customer_returns"
)

display(df_bronze)

response_id,age_group,gender,purchase_frequency,ever_returned,return_count,main_return_reason,wanted_exchange,exchange_available,replacement_unavailable,what_happened_when_unavailable,ingested_at
1,46-55,Female,Once every 3 months,Yes,1,Wrong Size,Yes,No,No Stock in Desired Size,Returned the Product,2026-08-26T14:19:57.676Z
2,26-35,Non-binary,2-3 times a month,Yes,2,Wrong Size,Yes,No,No Stock in Desired Size,Returned the Product,2026-08-26T14:19:57.676Z
3,56+,Male,Rarely,Yes,3,Wrong Size,Yes,No,No Stock in Desired Size,Returned the Product,2026-08-26T14:19:57.676Z
4,36-45,Female,Once every 3 months,Yes,4,Wrong Size,Yes,Yes,No Stock in Desired Size,Not Applicable,2026-08-26T14:19:57.676Z
5,18-25,Non-binary,2-3 times a month,Yes,5,Wrong Size,Yes,No,No Stock in Desired Size,Returned the Product,2026-08-26T14:19:57.676Z
6,46-55,female,Rarely,Yes,1,Wrong Size,Yes,No,No Stock in Desired Color,Waited for Restock,2026-08-26T14:19:57.676Z
7,26-35,Female,Once every 3 months,Yes,2,Wrong Size,Yes,No,No Stock in Desired Color,Waited for Restock,2026-08-26T14:19:57.676Z
8,56+,null,2-3 times a month,Yes,3,Wrong Size,No,Not Applicable,Not Applicable,Not Applicable,2026-08-26T14:19:57.676Z
9,36-45,Male,Rarely,Yes,4,Wrong Size,Yes,No,Other,Kept the Product,2026-08-26T14:19:57.676Z
10,18-25,Female,Once every 3 months,Yes,5,Wrong Size,Yes,No,No Stock in Desired Size,Returned the Product,2026-08-26T14:19:57.676Z


##**NULL check**

In [0]:
null_counts = df_bronze.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_bronze.columns
])

display(null_counts)

response_id,age_group,gender,purchase_frequency,ever_returned,return_count,main_return_reason,wanted_exchange,exchange_available,replacement_unavailable,what_happened_when_unavailable,ingested_at
0,1,2,2,0,1,1,1,0,1,1,0


In [0]:
display(
    df_bronze.filter(
        col("age_group").isNull() |
        col("gender").isNull() |
        col("purchase_frequency").isNull() |
        col("return_count").isNull() |
        col("main_return_reason").isNull() |
        col("wanted_exchange").isNull() |
        col("replacement_unavailable").isNull() |
        col("what_happened_when_unavailable").isNull()
    )
)

response_id,age_group,gender,purchase_frequency,ever_returned,return_count,main_return_reason,wanted_exchange,exchange_available,replacement_unavailable,what_happened_when_unavailable,ingested_at
8,56+,null,2-3 times a month,Yes,3,Wrong Size,No,Not Applicable,Not Applicable,Not Applicable,2026-08-26T14:19:57.676Z
19,36-45,Female,null,Yes,4,Wrong Size,Yes,No,Other,Kept the Product,2026-08-26T14:19:57.676Z
32,26-35,Non-binary,2-3 times a month,Yes,2,null,No,Not Applicable,Not Applicable,Not Applicable,2026-08-26T14:19:57.676Z
48,56+,Male,Rarely,Yes,3,Wrong Size,null,Not Applicable,Not Applicable,Not Applicable,2026-08-26T14:19:57.676Z
67,26-35,Female,Once every 3 months,Yes,2,Wrong Color,Yes,No,null,Waited for Restock,2026-08-26T14:19:57.676Z
84,36-45,Male,Rarely,Yes,null,Damaged Product,Yes,Yes,Not Applicable,Not Applicable,2026-08-26T14:19:57.676Z
85,null,Female,Once every 3 months,Yes,5,Damaged Product,No,Not Applicable,Not Applicable,Not Applicable,2026-08-26T14:19:57.676Z
102,26-35,Male,Rarely,Yes,2,Other,Yes,No,No Stock in Desired Size,null,2026-08-26T14:19:57.676Z
120,18-25,null,Rarely,No,0,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,2026-08-26T14:19:57.676Z
134,36-45,Non-binary,null,No,0,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,2026-08-26T14:19:57.676Z


##**Duplicate check**

In [0]:
duplicate_count = (
    df_bronze
    .groupBy(df_bronze.columns)
    .count()
    .filter(col("count") > 1)
    .count()
)

print("Duplicate records:", duplicate_count)

Duplicate records: 5


In [0]:
display(
    df_bronze
    .groupBy("age_group")
    .count()
    .orderBy("age_group")
)

age_group,count
null,1
26-35,1
18-25,29
26-35,29
36-45,31
46-55,30
56+,29


In [0]:
display(
    df_bronze
    .groupBy("gender")
    .count()
    .orderBy("gender")
)

gender,count
null,2
,1
Female,51
Male,46
Non-binary,48
Unknown,1
female,1


In [0]:
display(
    df_bronze
    .groupBy("purchase_frequency")
    .count()
    .orderBy("purchase_frequency")
)

purchase_frequency,count
null,2
2-3 times a month,49
Once a month,1
Once every 3 months,49
Rarely,48
Sometimes,1


In [0]:
display(
    df_bronze
    .groupBy("main_return_reason")
    .count()
    .orderBy("main_return_reason")
)

main_return_reason,count
null,1
Changed Mind,4
Damaged Product,6
Late Delivery,3
Not Applicable,43
Other,6
Poor Quality,7
Product Didn't Match Description,6
Wrong Color,23
Wrong Size,48


In [0]:
display(
    df_bronze
    .groupBy("replacement_unavailable")
    .count()
    .orderBy("replacement_unavailable")
)

replacement_unavailable,count
null,1
Exchange Not Offered,4
No Stock in Desired Color,11
No Stock in Desired Size,42
No stock in Desired Size,1
Not Applicable,82
Other,9


In [0]:
df_silver = df_bronze.dropDuplicates()

print("Bronze rows:", df_bronze.count())
print("Silver rows after removing duplicates:", df_silver.count())

Bronze rows: 150
Silver rows after removing duplicates: 145


In [0]:
from pyspark.sql.functions import trim
from pyspark.sql.functions import when

In [0]:
df_silver = (
    df_silver
    .withColumn("age_group", trim(col("age_group")))
    .withColumn("gender", trim(col("gender")))
    .withColumn("purchase_frequency", trim(col("purchase_frequency")))
    .withColumn("main_return_reason", trim(col("main_return_reason")))
    .withColumn("wanted_exchange", trim(col("wanted_exchange")))
    .withColumn("exchange_available", trim(col("exchange_available")))
    .withColumn("replacement_unavailable", trim(col("replacement_unavailable")))
    .withColumn(
        "what_happened_when_unavailable",
        trim(col("what_happened_when_unavailable"))
    )
)

In [0]:
display(
    df_silver
    .groupBy("age_group")
    .count()
    .orderBy("age_group")
)

age_group,count
null,1
18-25,28
26-35,30
36-45,30
46-55,29
56+,27


In [0]:
df_silver = (
    df_silver
    .withColumn(
        "gender",
        when(col("gender") == "female", "Female")
        .otherwise(col("gender"))
    )
    .withColumn(
        "main_return_reason",
        when(col("main_return_reason") == "Wrong Szie", "Wrong Size")
        .when(col("main_return_reason") == "wrong size", "Wrong Size")
        .when(col("main_return_reason") == "Wrong colour", "Wrong Color")
        .otherwise(col("main_return_reason"))
    )
    .withColumn(
        "replacement_unavailable",
        when(
            col("replacement_unavailable") == "No stock in Desired Size",
            "No Stock in Desired Size"
        )
        .otherwise(col("replacement_unavailable"))
    )
)

In [0]:
display(
    df_silver
    .groupBy("main_return_reason")
    .count()
    .orderBy("main_return_reason")
)

main_return_reason,count
null,1
Changed Mind,4
Damaged Product,6
Late Delivery,3
Not Applicable,43
Other,5
Poor Quality,7
Product Didn't Match Description,5
Wrong Color,24
Wrong Size,47


In [0]:
display(
    df_silver
    .groupBy("replacement_unavailable")
    .count()
    .orderBy("replacement_unavailable")
)

replacement_unavailable,count
null,1
Exchange Not Offered,4
No Stock in Desired Color,11
No Stock in Desired Size,41
Not Applicable,79
Other,9


In [0]:
display(
    df_silver
    .filter(
        (col("age_group") == "") |
        (col("gender") == "") |
        (col("purchase_frequency") == "") |
        (col("main_return_reason") == "") |
        (col("wanted_exchange") == "") |
        (col("replacement_unavailable") == "") |
        (col("what_happened_when_unavailable") == "")
    )
)

response_id,age_group,gender,purchase_frequency,ever_returned,return_count,main_return_reason,wanted_exchange,exchange_available,replacement_unavailable,what_happened_when_unavailable,ingested_at
44,36-45,,2-3 times a month,Yes,4,Wrong Size,Yes,Yes,Not Applicable,Not Applicable,2026-08-26T14:19:57.676Z


In [0]:
from pyspark.sql.functions import trim, when, col

df_silver = (
    df_silver
    .withColumn(
        "gender",
        when(trim(col("gender")) == "", None)
        .otherwise(col("gender"))
    )
    .withColumn(
        "age_group",
        when(trim(col("age_group")) == "", None)
        .otherwise(col("age_group"))
    )
    .withColumn(
        "purchase_frequency",
        when(trim(col("purchase_frequency")) == "", None)
        .otherwise(col("purchase_frequency"))
    )
    .withColumn(
        "main_return_reason",
        when(trim(col("main_return_reason")) == "", None)
        .otherwise(col("main_return_reason"))
    )
    .withColumn(
        "wanted_exchange",
        when(trim(col("wanted_exchange")) == "", None)
        .otherwise(col("wanted_exchange"))
    )
    .withColumn(
        "replacement_unavailable",
        when(trim(col("replacement_unavailable")) == "", None)
        .otherwise(col("replacement_unavailable"))
    )
    .withColumn(
        "what_happened_when_unavailable",
        when(trim(col("what_happened_when_unavailable")) == "", None)
        .otherwise(col("what_happened_when_unavailable"))
    )
)

In [0]:
df_silver = (
    df_silver
    .fillna({
        "age_group": "Unknown",
        "gender": "Unknown",
        "purchase_frequency": "Unknown",
        "main_return_reason": "Unknown",
        "wanted_exchange": "Unknown",
        "replacement_unavailable": "Unknown",
        "what_happened_when_unavailable": "Unknown"
    })
)

In [0]:
display(
    df_silver.select([
        count(when(col(c).isNull(), c)).alias(c)
        for c in df_silver.columns
    ])
)

response_id,age_group,gender,purchase_frequency,ever_returned,return_count,main_return_reason,wanted_exchange,exchange_available,replacement_unavailable,what_happened_when_unavailable,ingested_at
0,0,0,0,0,1,0,0,0,0,0,0


In [0]:
display(
    df_silver
    .filter(col("return_count").isNull())
)

response_id,age_group,gender,purchase_frequency,ever_returned,return_count,main_return_reason,wanted_exchange,exchange_available,replacement_unavailable,what_happened_when_unavailable,ingested_at
84,36-45,Male,Rarely,Yes,null,Damaged Product,Yes,Yes,Not Applicable,Not Applicable,2026-08-26T14:19:57.676Z


In [0]:


df_silver = df_silver.withColumn(
    "return_count_missing",
    when(col("return_count").isNull(), 1).otherwise(0)
)

display(
    df_silver.select(
        "response_id",
        "ever_returned",
        "return_count",
        "return_count_missing"
    )
    .filter(col("return_count_missing") == 1)
)

response_id,ever_returned,return_count,return_count_missing
84,Yes,null,1


In [0]:
display(
    df_silver
    .filter(
        (col("ever_returned") == "No") &
        (col("return_count") > 0)
    )
)

response_id,age_group,gender,purchase_frequency,ever_returned,return_count,main_return_reason,wanted_exchange,exchange_available,replacement_unavailable,what_happened_when_unavailable,ingested_at,return_count_missing
127,26-35,Female,Once every 3 months,No,2,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,2026-08-26T14:19:57.676Z,0


In [0]:
df_silver = df_silver.withColumn(
    "return_count",
    when(
        col("ever_returned") == "No",
        0
    ).otherwise(col("return_count"))
)

display(
    df_silver.filter(
        (col("ever_returned") == "No") &
        (col("return_count") > 0)
    )
)


response_id,age_group,gender,purchase_frequency,ever_returned,return_count,main_return_reason,wanted_exchange,exchange_available,replacement_unavailable,what_happened_when_unavailable,ingested_at,return_count_missing


In [0]:
display(
    df_silver.filter(
        (col("ever_returned") == "No") &
        (col("wanted_exchange") != "Not Applicable")
    )
)



response_id,age_group,gender,purchase_frequency,ever_returned,return_count,main_return_reason,wanted_exchange,exchange_available,replacement_unavailable,what_happened_when_unavailable,ingested_at,return_count_missing


In [0]:
display(
    df_silver
    .groupBy(
        "wanted_exchange",
        "exchange_available",
        "replacement_unavailable"
    )
    .count()
    .orderBy(
        "wanted_exchange",
        "exchange_available",
        "replacement_unavailable"
    )
)

wanted_exchange,exchange_available,replacement_unavailable,count
Maybe,Not Applicable,Not Applicable,1
No,Not Applicable,No Stock in Desired Size,1
No,Not Applicable,Not Applicable,27
Not Applicable,Not Applicable,Not Applicable,42
Not Applicable,Unavailable,Not Applicable,1
Unknown,Not Applicable,Not Applicable,1
YES,No,No Stock in Desired Size,1
Yes,No,Exchange Not Offered,4
Yes,No,No Stock in Desired Color,10
Yes,No,No Stock in Desired Size,38


In [0]:
df_silver = (
    df_silver
    .withColumn(
        "wanted_exchange",
        when(col("wanted_exchange") == "YES", "Yes")
        .otherwise(col("wanted_exchange"))
    )
    .withColumn(
        "exchange_available",
        when(col("exchange_available") == "no", "No")
        .when(col("exchange_available") == "Unavailable", "No")
        .otherwise(col("exchange_available"))
    )
)



df_silver = (
    df_silver
    .withColumn(
        "replacement_unavailable",
        when(
            col("wanted_exchange") == "No",
            "Not Applicable"
        )
        .otherwise(col("replacement_unavailable"))
    )
    .withColumn(
        "exchange_available",
        when(
            col("replacement_unavailable").isin(
                "No Stock in Desired Size",
                "No Stock in Desired Color",
                "Exchange Not Offered",
                "Other"
            ),
            "No"
        )
        .otherwise(col("exchange_available"))
    )
)

In [0]:
display(
    df_silver
    .groupBy(
        "wanted_exchange",
        "exchange_available",
        "replacement_unavailable"
    )
    .count()
    .orderBy(
        "wanted_exchange",
        "exchange_available",
        "replacement_unavailable"
    )
)

wanted_exchange,exchange_available,replacement_unavailable,count
Maybe,Not Applicable,Not Applicable,1
No,Not Applicable,Not Applicable,28
Not Applicable,No,Not Applicable,1
Not Applicable,Not Applicable,Not Applicable,42
Unknown,Not Applicable,Not Applicable,1
Yes,No,Exchange Not Offered,4
Yes,No,No Stock in Desired Color,11
Yes,No,No Stock in Desired Size,40
Yes,No,Other,9
Yes,No,Unknown,1


In [0]:
df_silver = df_silver.withColumn(
    "exchange_available",
    when(
        col("wanted_exchange") == "Not Applicable",
        "Not Applicable"
    ).otherwise(col("exchange_available"))
)

In [0]:
display(
    df_silver
    .groupBy(
        "wanted_exchange",
        "exchange_available",
        "replacement_unavailable"
    )
    .count()
)

wanted_exchange,exchange_available,replacement_unavailable,count
Yes,No,No Stock in Desired Size,40
Yes,No,No Stock in Desired Color,11
No,Not Applicable,Not Applicable,28
Yes,No,Other,9
Yes,Yes,Not Applicable,7
Yes,No,Exchange Not Offered,4
Unknown,Not Applicable,Not Applicable,1
Yes,No,Unknown,1
Maybe,Not Applicable,Not Applicable,1
Not Applicable,Not Applicable,Not Applicable,43


In [0]:
print("Silver rows:", df_silver.count())

Silver rows: 145


In [0]:
display(
    df_silver.select([
        count(
            when(col(c).isNull(), c)
        ).alias(c)
        for c in df_silver.columns
    ])
)

response_id,age_group,gender,purchase_frequency,ever_returned,return_count,main_return_reason,wanted_exchange,exchange_available,replacement_unavailable,what_happened_when_unavailable,ingested_at,return_count_missing
0,0,0,0,0,1,0,0,0,0,0,0,0


In [0]:
print(
    "Duplicate rows:",
    df_silver.count() - df_silver.dropDuplicates().count()
)

Duplicate rows: 0


In [0]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(
        "customer_returns.silver.slv_customer_returns"
    )

In [0]:
df_silver_check = spark.table(
    "customer_returns.silver.slv_customer_returns"
)

print("Silver rows:", df_silver_check.count())

display(df_silver_check)

Silver rows: 145


response_id,age_group,gender,purchase_frequency,ever_returned,return_count,main_return_reason,wanted_exchange,exchange_available,replacement_unavailable,what_happened_when_unavailable,ingested_at,return_count_missing
37,26-35,Female,Once every 3 months,Yes,2,Wrong Size,Yes,No,No Stock in Desired Color,Waited for Restock,2026-08-26T14:19:57.676Z,0
42,26-35,Male,Rarely,Yes,2,Wrong Size,Yes,No,No Stock in Desired Size,Returned the Product,2026-08-26T14:19:57.676Z,0
59,36-45,Non-binary,2-3 times a month,Yes,4,Wrong Size,Yes,No,Other,Kept the Product,2026-08-26T14:19:57.676Z,0
66,46-55,Male,Rarely,Yes,1,Wrong Color,Yes,No,No Stock in Desired Color,Waited for Restock,2026-08-26T14:19:57.676Z,0
71,46-55,Non-binary,2-3 times a month,Yes,1,Wrong Color,Yes,No,No Stock in Desired Size,Returned the Product,2026-08-26T14:19:57.676Z,0
88,56+,Female,Once every 3 months,Yes,3,Damaged Product,No,Not Applicable,Not Applicable,Not Applicable,2026-08-26T14:19:57.676Z,0
103,56+,Female,Once every 3 months,Yes,3,Other,No,Not Applicable,Not Applicable,Not Applicable,2026-08-26T14:19:57.676Z,0
15,18-25,Unknown,Rarely,Yes,5,Wrong Size,Yes,No,No Stock in Desired Size,Returned the Product,2026-08-26T14:19:57.676Z,0
82,26-35,Female,Once every 3 months,Yes,2,Poor Quality,No,Not Applicable,Not Applicable,Not Applicable,2026-08-26T14:19:57.676Z,0
83,56+,Non-binary,2-3 times a month,Yes,3,Damaged Product,No,Not Applicable,Not Applicable,Not Applicable,2026-08-26T14:19:57.676Z,0


In [0]:
df_gold = spark.table(
    "customer_returns.silver.slv_customer_returns"
)

display(df_gold)

response_id,age_group,gender,purchase_frequency,ever_returned,return_count,main_return_reason,wanted_exchange,exchange_available,replacement_unavailable,what_happened_when_unavailable,ingested_at,return_count_missing
37,26-35,Female,Once every 3 months,Yes,2,Wrong Size,Yes,No,No Stock in Desired Color,Waited for Restock,2026-08-26T14:19:57.676Z,0
42,26-35,Male,Rarely,Yes,2,Wrong Size,Yes,No,No Stock in Desired Size,Returned the Product,2026-08-26T14:19:57.676Z,0
59,36-45,Non-binary,2-3 times a month,Yes,4,Wrong Size,Yes,No,Other,Kept the Product,2026-08-26T14:19:57.676Z,0
66,46-55,Male,Rarely,Yes,1,Wrong Color,Yes,No,No Stock in Desired Color,Waited for Restock,2026-08-26T14:19:57.676Z,0
71,46-55,Non-binary,2-3 times a month,Yes,1,Wrong Color,Yes,No,No Stock in Desired Size,Returned the Product,2026-08-26T14:19:57.676Z,0
88,56+,Female,Once every 3 months,Yes,3,Damaged Product,No,Not Applicable,Not Applicable,Not Applicable,2026-08-26T14:19:57.676Z,0
103,56+,Female,Once every 3 months,Yes,3,Other,No,Not Applicable,Not Applicable,Not Applicable,2026-08-26T14:19:57.676Z,0
15,18-25,Unknown,Rarely,Yes,5,Wrong Size,Yes,No,No Stock in Desired Size,Returned the Product,2026-08-26T14:19:57.676Z,0
82,26-35,Female,Once every 3 months,Yes,2,Poor Quality,No,Not Applicable,Not Applicable,Not Applicable,2026-08-26T14:19:57.676Z,0
83,56+,Non-binary,2-3 times a month,Yes,3,Damaged Product,No,Not Applicable,Not Applicable,Not Applicable,2026-08-26T14:19:57.676Z,0


In [0]:
total_customers = df_gold.count()

returned_customers = (
    df_gold
    .filter(col("ever_returned") == "Yes")
    .count()
)

not_returned_customers = (
    df_gold
    .filter(col("ever_returned") == "No")
    .count()
)

return_rate = returned_customers / total_customers * 100

print("Total customers:", total_customers)
print("Returned customers:", returned_customers)
print("Not returned customers:", not_returned_customers)
print("Return rate:", round(return_rate, 2), "%")

Total customers: 145
Returned customers: 102
Not returned customers: 43
Return rate: 70.34 %


In [0]:
display(
    df_gold
    .filter(col("ever_returned") == "Yes")
    .groupBy("main_return_reason")
    .count()
    .orderBy(col("count").desc())
)

main_return_reason,count
Wrong Size,47
Wrong Color,24
Poor Quality,7
Damaged Product,6
Product Didn't Match Description,5
Other,5
Changed Mind,4
Late Delivery,3
Unknown,1


In [0]:
display(
    df_gold
    .filter(col("wanted_exchange") == "Yes")
    .groupBy("replacement_unavailable")
    .count()
    .orderBy(col("count").desc())
)

replacement_unavailable,count
No Stock in Desired Size,40
No Stock in Desired Color,11
Other,9
Not Applicable,7
Exchange Not Offered,4
Unknown,1


In [0]:
display(
    df_gold
    .filter(
        (col("ever_returned") == "Yes") &
        (col("main_return_reason") == "Wrong Size")
    )
    .groupBy(
        "wanted_exchange",
        "exchange_available",
        "replacement_unavailable"
    )
    .count()
    .orderBy(col("count").desc())
)

wanted_exchange,exchange_available,replacement_unavailable,count
Yes,No,No Stock in Desired Size,23
Yes,No,No Stock in Desired Color,8
Yes,Yes,Not Applicable,5
Yes,No,Other,5
No,Not Applicable,Not Applicable,3
Yes,No,Exchange Not Offered,2
Unknown,Not Applicable,Not Applicable,1


In [0]:
smart_exchange_candidates = (
    df_gold
    .filter(
        (col("wanted_exchange") == "Yes") &
        col("replacement_unavailable").isin(
            "No Stock in Desired Size",
            "No Stock in Desired Color"
        )
    )
)

print(
    "Smart Exchange candidates:",
    smart_exchange_candidates.count()
)

Smart Exchange candidates: 51


In [0]:
df_gold = df_gold.withColumn(
    "smart_exchange_candidate",
    when(
        (col("wanted_exchange") == "Yes") &
        col("replacement_unavailable").isin(
            "No Stock in Desired Size",
            "No Stock in Desired Color"
        ),
        "Yes"
    ).otherwise("No")
)

display(
    df_gold
    .groupBy("smart_exchange_candidate")
    .count()
)

smart_exchange_candidate,count
Yes,51
No,94


In [0]:
exchange_customers = (
    df_gold
    .filter(col("wanted_exchange") == "Yes")
    .count()
)

smart_exchange_count = (
    df_gold
    .filter(col("smart_exchange_candidate") == "Yes")
    .count()
)

smart_exchange_rate = (
    smart_exchange_count / exchange_customers * 100
)

print("Customers wanting exchange:", exchange_customers)
print("Smart Exchange candidates:", smart_exchange_count)
print(
    "Smart Exchange candidate rate:",
    round(smart_exchange_rate, 2),
    "%"
)

Customers wanting exchange: 72
Smart Exchange candidates: 51
Smart Exchange candidate rate: 70.83 %


In [0]:
df_exchange_summary = (
    df_gold
    .filter(col("wanted_exchange") == "Yes")
    .groupBy("replacement_unavailable")
    .count()
    .withColumnRenamed("count", "customer_count")
)

display(
    df_exchange_summary
    .orderBy(col("customer_count").desc())
)

replacement_unavailable,customer_count
No Stock in Desired Size,40
No Stock in Desired Color,11
Other,9
Not Applicable,7
Exchange Not Offered,4
Unknown,1


In [0]:
df_return_exchange = (
    df_gold
    .filter(
        (col("ever_returned") == "Yes") &
        (col("wanted_exchange") == "Yes")
    )
    .groupBy(
        "main_return_reason",
        "replacement_unavailable"
    )
    .count()
    .withColumnRenamed("count", "customer_count")
)

display(
    df_return_exchange
    .orderBy(
        col("main_return_reason"),
        col("customer_count").desc()
    )
)

main_return_reason,replacement_unavailable,customer_count
Changed Mind,Not Applicable,1
Damaged Product,No Stock in Desired Color,1
Damaged Product,Not Applicable,1
Late Delivery,Other,1
Other,No Stock in Desired Size,2
Poor Quality,No Stock in Desired Size,2
Poor Quality,Exchange Not Offered,1
Product Didn't Match Description,No Stock in Desired Size,2
Wrong Color,No Stock in Desired Size,11
Wrong Color,Other,3


In [0]:
df_gold = df_gold.withColumn(
    "smart_exchange_type",
    when(
        col("wanted_exchange") == "Yes",
        when(
            col("replacement_unavailable") == "No Stock in Desired Size",
            "Size Availability"
        )
        .when(
            col("replacement_unavailable") == "No Stock in Desired Color",
            "Color Availability"
        )
        .otherwise("Other Exchange Issue")
    )
    .otherwise("Not a Candidate")
)

In [0]:
display(
    df_gold
    .groupBy("smart_exchange_type")
    .count()
    .orderBy(col("count").desc())
)

smart_exchange_type,count
Not a Candidate,73
Size Availability,40
Other Exchange Issue,21
Color Availability,11


In [0]:
df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(
        "customer_returns.gold.gold_customer_returns"
    )

In [0]:
df_return_summary = (
    df_gold
    .filter(col("ever_returned") == "Yes")
    .groupBy("main_return_reason")
    .count()
    .withColumnRenamed("count", "return_count")
    .orderBy(col("return_count").desc())
)

display(df_return_summary)

main_return_reason,return_count
Wrong Size,47
Wrong Color,24
Poor Quality,7
Damaged Product,6
Product Didn't Match Description,5
Other,5
Changed Mind,4
Late Delivery,3
Unknown,1


In [0]:
df_return_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "customer_returns.gold.gold_return_summary"
    )

display(
    spark.table(
        "customer_returns.gold.gold_return_summary"
    )
)

main_return_reason,return_count
Wrong Size,47
Wrong Color,24
Poor Quality,7
Damaged Product,6
Other,5
Product Didn't Match Description,5
Changed Mind,4
Late Delivery,3
Unknown,1


In [0]:
df_exchange_summary = (
    df_gold
    .filter(col("wanted_exchange") == "Yes")
    .groupBy("replacement_unavailable")
    .count()
    .withColumnRenamed("count", "customer_count")
    .orderBy(col("customer_count").desc())
)

display(df_exchange_summary)

replacement_unavailable,customer_count
No Stock in Desired Size,40
No Stock in Desired Color,11
Other,9
Not Applicable,7
Exchange Not Offered,4
Unknown,1


In [0]:
df_exchange_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "customer_returns.gold.gold_exchange_summary"
    )

In [0]:
display(
    spark.table(
        "customer_returns.gold.gold_exchange_summary"
    )
)

replacement_unavailable,customer_count
No Stock in Desired Size,40
No Stock in Desired Color,11
Other,9
Not Applicable,7
Exchange Not Offered,4
Unknown,1


In [0]:
df_reason_exchange_summary = (
    df_gold
    .filter(
        (col("ever_returned") == "Yes") &
        (col("wanted_exchange") == "Yes")
    )
    .groupBy(
        "main_return_reason",
        "replacement_unavailable"
    )
    .count()
    .withColumnRenamed("count", "customer_count")
    .orderBy(
        col("main_return_reason"),
        col("customer_count").desc()
    )
)

display(df_reason_exchange_summary)

main_return_reason,replacement_unavailable,customer_count
Changed Mind,Not Applicable,1
Damaged Product,No Stock in Desired Color,1
Damaged Product,Not Applicable,1
Late Delivery,Other,1
Other,No Stock in Desired Size,2
Poor Quality,No Stock in Desired Size,2
Poor Quality,Exchange Not Offered,1
Product Didn't Match Description,No Stock in Desired Size,2
Wrong Color,No Stock in Desired Size,11
Wrong Color,Other,3


In [0]:
df_reason_exchange_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "customer_returns.gold.gold_reason_exchange_summary"
    )
display(
    spark.table(
        "customer_returns.gold.gold_reason_exchange_summary"
    )
)

main_return_reason,replacement_unavailable,customer_count
Changed Mind,Not Applicable,1
Damaged Product,Not Applicable,1
Damaged Product,No Stock in Desired Color,1
Late Delivery,Other,1
Other,No Stock in Desired Size,2
Poor Quality,No Stock in Desired Size,2
Poor Quality,Exchange Not Offered,1
Product Didn't Match Description,No Stock in Desired Size,2
Wrong Color,No Stock in Desired Size,11
Wrong Color,Other,3


In [0]:
print("Customer Gold:",
      spark.table("customer_returns.gold.gold_customer_returns").count())

print("Return Summary:",
      spark.table("customer_returns.gold.gold_return_summary").count())

print("Exchange Summary:",
      spark.table("customer_returns.gold.gold_exchange_summary").count())

print("Reason × Exchange:",
      spark.table("customer_returns.gold.gold_reason_exchange_summary").count())

Customer Gold: 145
Return Summary: 9
Exchange Summary: 6
Reason × Exchange: 18


In [0]:
display(df_gold)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-7840526560569385>, line 1
----> 1 display(df_gold)

NameError: name 'df_gold' is not defined